# BikeEase Incremental Capstone Project - Part 4

# Install NLTK (Natural Language Toolkit)

In [1]:
!pip install nltk

# Install Transformers, Datasets, Torch

In [2]:
!pip install transformers datasets torch

# Libraries Imports

In [3]:
import pandas as pd
import numpy as np
import nltk
import string
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Sequential

from tf_keras.callbacks import EarlyStopping

from datasets import Dataset

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

# Download NLTK Resources

In [4]:
nltk.download('punkt_tab')    # Download punkt_tab resource
nltk.download('stopwords')    # Common stopwords
nltk.download('wordnet')      # WordNet lexical database

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Data Collection & Preprocessing

## Load bike_rental_reviews.csv Dataset into a DataFrame using Pandas

In [5]:
og = pd.read_csv('bike_rental_reviews.csv')
df = og.copy()

## Data Analysis

In [6]:
# view first few rows
df.head()

,review_text,sentiment
0,"The entire process was easy, and the availabil...",positive
1,Standard rental process. The mobile app was ac...,neutral
2,One of the best bike rentals I’ve had. The mob...,positive
3,One of the best bike rentals I’ve had. The cus...,positive
4,Not worth the money. The seat comfort was a ma...,negative


In [7]:
# shape
df.shape

# 50000 rows
# 2 columns

(50000, 2)

In [8]:
# initial statistics about DataFrame
df.describe()

,review_text,sentiment
count,50000,50000
unique,300,3
top,Worst experience ever. The mobile app ruined t...,negative
freq,201,16840


In [9]:
#view column names, and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  50000 non-null  object
 1   sentiment    50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [10]:
# identify missing values
df.isnull().sum()

# there are no missing values

,0
review_text,0
sentiment,0


In [11]:
# check for duplicate records
df.duplicated().sum()

np.int64(49700)

A large number of duplicated rows were found in the dataset, despite there being no missing values. To better understand this, the rows were grouped and inspected across all features to determine if they were truly identical.

In [12]:
duplicates_grouped = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))

duplicates_grouped

,review_text,sentiment
655,A decent ride but not the best availability.,neutral
873,A decent ride but not the best availability.,neutral
934,A decent ride but not the best availability.,neutral
1074,A decent ride but not the best availability.,neutral
1169,A decent ride but not the best availability.,neutral
...,...,...
49279,Worst experience ever. The support staff ruine...,negative
49318,Worst experience ever. The support staff ruine...,negative
49486,Worst experience ever. The support staff ruine...,negative
49628,Worst experience ever. The support staff ruine...,negative


The inspection showed that the rows were truly identical, which is unrealistic. It would suggest that many customers had the exact same reviews and feedbacks, which is highly unlikely. This could be a result of fake reviews.

As a result, the duplicated rows were removed to ensure data quality and prevent misleading the analysis.

In [13]:
# dropping duplicates
df_clean = df.drop_duplicates()

# checking for duplicates again to confirm deletion
df_clean.duplicated().sum()

df = df_clean.copy()

In [14]:
# shape
df.shape

# 300 rows
# 2 columns

(300, 2)

In [15]:
# data balance/imbalanceness
print(df['sentiment'].value_counts())

sentiment
positive    100
neutral     100
negative    100
Name: count, dtype: int64


## Text Cleaning

### Lowercasing

In [16]:
df['review_text'] = df['review_text'].str.lower()

df.head()

,review_text,sentiment
0,"the entire process was easy, and the availabil...",positive
1,standard rental process. the mobile app was ac...,neutral
2,one of the best bike rentals i’ve had. the mob...,positive
3,one of the best bike rentals i’ve had. the cus...,positive
4,not worth the money. the seat comfort was a ma...,negative


### Removing Punctuation

In [17]:
df['review_text'] = df['review_text'].str.replace(f"[{re.escape(string.punctuation)}]", "", regex=True)

df.head()

,review_text,sentiment
0,the entire process was easy and the availabili...,positive
1,standard rental process the mobile app was acc...,neutral
2,one of the best bike rentals i’ve had the mobi...,positive
3,one of the best bike rentals i’ve had the cust...,positive
4,not worth the money the seat comfort was a maj...,negative


### Tokenization

In [18]:
df['review_text'] = df['review_text'].apply(word_tokenize)

df.head()

,review_text,sentiment
0,"[the, entire, process, was, easy, and, the, av...",positive
1,"[standard, rental, process, the, mobile, app, ...",neutral
2,"[one, of, the, best, bike, rentals, i, ’, ve, ...",positive
3,"[one, of, the, best, bike, rentals, i, ’, ve, ...",positive
4,"[not, worth, the, money, the, seat, comfort, w...",negative


### Stopword Removal

In [19]:
stop_words = set(stopwords.words('english'))

df['review_text'] = df['review_text'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

df.head()

,review_text,sentiment
0,"[entire, process, easy, availability, high, qu...",positive
1,"[standard, rental, process, mobile, app, accep...",neutral
2,"[one, best, bike, rentals, ’, mobile, app, mad...",positive
3,"[one, best, bike, rentals, ’, customer, servic...",positive
4,"[worth, money, seat, comfort, major, letdown]",negative


### Lemmatization

In [20]:
lemmatizer = nltk.stem.WordNetLemmatizer()

df['review_text'] = df['review_text'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

df.head()

,review_text,sentiment
0,"[entire, process, easy, availability, high, qu...",positive
1,"[standard, rental, process, mobile, app, accep...",neutral
2,"[one, best, bike, rental, ’, mobile, app, made...",positive
3,"[one, best, bike, rental, ’, customer, service...",positive
4,"[worth, money, seat, comfort, major, letdown]",negative


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300 entries, 0 to 1634
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  300 non-null    object
 1   sentiment    300 non-null    object
dtypes: object(2)
memory usage: 7.0+ KB


## Save the processed dataset as 'bike_rental_features_cleaned.csv'

In [22]:
df.to_csv('bike_rental_features_cleaned.csv', index=False)

# Sentiment Analysis

In [23]:
# join the tokens back into strings
df['review_text'] = df['review_text'].apply(lambda tokens: ' '.join(tokens))

## Features and Target Selection

In [24]:
# select features and target
X = df['review_text']
y = df['sentiment']

## Training and Test sets Splitting

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Vectorization

In [26]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## Traditional Models

### Logistic Regression

In [27]:
# create the model object
log_model = LogisticRegression(max_iter=1000)

# fit and train model
log_model.fit(X_train_tfidf, y_train)

# predict on test set
y_test_pred_log = log_model.predict(X_test_tfidf)

# evaluation
print("Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_test, y_test_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred_log))

Logistic Regression Performance:
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

    negative       1.00      1.00      1.00        23
     neutral       1.00      1.00      1.00        17
    positive       1.00      1.00      1.00        20

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60


Confusion Matrix:
 [[23  0  0]
 [ 0 17  0]
 [ 0  0 20]]


### Naïve Bayes

In [28]:
# create the model object
nb_model = MultinomialNB()

# fit and train model
nb_model.fit(X_train_tfidf, y_train)

# predict on test set
y_test_pred_log = nb_model.predict(X_test_tfidf)

# evaluation
print("Naïve Bayes Performance:")
print("Accuracy:", accuracy_score(y_test, y_test_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred_log))

Naïve Bayes Performance:
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

    negative       1.00      1.00      1.00        23
     neutral       1.00      1.00      1.00        17
    positive       1.00      1.00      1.00        20

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60


Confusion Matrix:
 [[23  0  0]
 [ 0 17  0]
 [ 0  0 20]]


## Deep Learning Models

### Sentiment Mapping

In [29]:
# convert text labels to integers
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
df['sentiment'] = df['sentiment'].map(label_mapping)

### LSTM

In [30]:
# limit vocab size for efficiency
vocab_size = 5000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['review_text'])

# convert reviews into sequences of integers
sequences = tokenizer.texts_to_sequences(df['review_text'])

# pad sequences so all reviews have the same length
max_length = 10
X = pad_sequences(sequences, maxlen=max_length, padding="post", truncating="post")

# target labels
y = df['sentiment'].values

# hold out validation set
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X, y, test_size=30, random_state=42, stratify=y
)

# split data
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

model_lstm = Sequential([
    # embedding layer (turn words into dense vectors)
    layers.Embedding(input_dim=vocab_size, output_dim=64),

    # LSTM layer with 32 units (memory cells)
    layers.LSTM(32, return_sequences=False),

    layers.Dense(32, activation="relu"),

    layers.Dropout(0.3),

    # output layer (3 classes: negative, neutral, positive)
    layers.Dense(3, activation="softmax")
])

# compile model
model_lstm.compile(loss="sparse_categorical_crossentropy",
                   optimizer="adam",
                   metrics=["accuracy"])

# early stopping
early_stop = EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True
)

# train model
history = model_lstm.fit(
    X_train, y_train,
    epochs=20,
    validation_data=(X_val, y_val),
    batch_size=16,
    callbacks=[early_stop]
)

# evaluation
loss, accuracy = model_lstm.evaluate(X_holdout, y_holdout, verbose=0)
print(f"\nFinal Hold-Out Test Accuracy: {accuracy:.2f}")

# predictions
y_pred = model_lstm.predict(X_holdout).argmax(axis=1)

# detailed evaluation
print("\nClassification Report (Hold-Out Set):")
print(classification_report(y_holdout, y_pred, target_names=["negative", "neutral", "positive"]))

print("\nConfusion Matrix (Hold-Out Set):")
print(confusion_matrix(y_holdout, y_pred))

Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.3302 - loss: 1.0984 - val_accuracy: 0.5370 - val_loss: 1.0935
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4993 - loss: 1.0888 - val_accuracy: 0.5741 - val_loss: 1.0749
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6454 - loss: 1.0425 - val_accuracy: 0.5185 - val_loss: 0.9811
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6314 - loss: 0.8802 - val_accuracy: 0.7222 - val_loss: 0.6486
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7756 - loss: 0.5400 - val_accuracy: 1.0000 - val_loss: 0.3116
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9927 - loss: 0.2403 - val_accuracy: 1.0000 - val_loss: 0.0512
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0672 - val_accuracy: 1.0000 - val_loss: 0.0073
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0145 - val_accuracy: 1.0000 - val_lo

### Transformers (BERT)

In [31]:
# split text and labels for BERT
X_train_text, X_holdout_text, y_train, y_holdout = train_test_split(
    df['review_text'], df['sentiment'], test_size=30, random_state=42, stratify=df['sentiment']
)

# further split training into train/dev for early stopping
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_train_text, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# prepare datasets
train_dataset = Dataset.from_dict({"text": X_train_text.tolist(), "label": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test_text.tolist(), "label": y_test.tolist()})
validation_dataset = Dataset.from_dict({"text": X_holdout_text.tolist(), "label": y_holdout.tolist()})

# tokenization with BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# tokenize the dataset
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
validation_dataset = validation_dataset.map(tokenize, batched=True)

# set format for PyTorch
columns = ["input_ids", "attention_mask", "label"]
train_dataset.set_format("torch", columns=columns)
test_dataset.set_format("torch", columns=columns)
validation_dataset.set_format("torch", columns=columns)

# load BERT model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

# metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# trainer arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer.train()

# predict
predictions = trainer.predict(validation_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = y_holdout

# evaluation
print("\n Accuracy:", accuracy_score(y_true, y_pred))
print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))
print("\n Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3024287785.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.012500,0.598859,0.944444,0.946394,0.944444,0.944931
2,0.312300,0.041758,1.000000,1.000000,1.000000,1.000000
3,0.021400,0.004358,1.000000,1.000000,1.000000,1.000000
4,0.003800,0.001805,1.000000,1.000000,1.000000,1.000000
5,0.002000,0.001226,1.000000,1.000000,1.000000,1.000000
6,0.001600,0.001015,1.000000,1.000000,1.000000,1.000000
7,0.001300,0.000893,1.000000,1.000000,1.000000,1.000000



 Accuracy: 1.0

 Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        10
     neutral       1.00      1.00      1.00        10
    positive       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


 Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]


# Report

This phase of the BikeEase project focuses on analyzing customer reviews. The goal is to build an NLP-powered sentiment analysis system that automatically classifies reviews as positive, neutral, or negative and identifies key themes in customer feedback. This will help BikeEase understand customer sentiment, uncover pain points, and improve their services.

The BikeEase sentiment dataset started off with 50,000 rows, but many of them were exact duplicates. That doesn't make much sense since it is unlikely that so many customers wrote exactly the same reviews, and it could be due to fake entries. To make the analysis more reliable, all duplicate rows were removed, leaving just 300 examples. The final dataset is perfectly balanced, with 100 reviews each for positive, neutral, and negative sentiments.

Four models were tested: Logistic Regression, Naïve Bayes, LSTM, and BERT. All of them achieved perfect accuracy on their respective test or hold-out sets. This shows that the dataset is fairly simple and even TF-IDF-based models handle it well. The LSTM and BERT models, while capable of capturing more complex text patterns, did not provide any extra advantage given the small and clean dataset.

Overall, for small and balanced datasets like this, simple models are fast and effective. Deep learning models aren't really needed, but they could be more useful for bigger, messier, or more complex datasets.